In [5]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/fraud-project/'
print("Drive connected!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive connected!


In [6]:
!pip install imbalanced-learn -q

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

print("Ready!")

Ready!


In [7]:
train    = pd.read_csv(DRIVE_PATH + 'train_transaction.csv')
identity = pd.read_csv(DRIVE_PATH + 'train_identity.csv')
df       = train.merge(identity, on='TransactionID', how='left')
print(f"Merged shape: {df.shape}")

Merged shape: (590540, 434)


In [8]:
#  Drop High-Missing Columns
threshold   = 0.5
missing_pct = df.isnull().sum() / len(df)
cols_drop   = missing_pct[missing_pct > threshold].index.tolist()
df          = df.drop(columns=cols_drop)

print(f"Dropped {len(cols_drop)} columns")
print(f"Shape now: {df.shape}")

Dropped 214 columns
Shape now: (590540, 220)


In [9]:
# Missing Values
num_cols = df.select_dtypes(include=['float64','int64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print(f"Missing remaining: {df.isnull().sum().sum()}")

Missing remaining: 0


In [11]:
# Encode Categoricals
le = LabelEncoder()
for col in cat_cols:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))

print(f"Encoded {len(cat_cols)} columns")

Encoded 9 columns


In [12]:
# Feature Engineering
df['hour']       = (df['TransactionDT'] // 3600) % 24
df['day']        = (df['TransactionDT'] // (3600*24)) % 7
df['is_night']   = (df['hour'] <= 6).astype(int)
df['is_weekend'] = (df['day'] >= 5).astype(int)
df['amt_log']    = np.log1p(df['TransactionAmt'])
df['is_round']   = (df['TransactionAmt'] % 1 == 0).astype(int)
df['card1_freq'] = df.groupby('card1')['card1'].transform('count')

print(f"Features added. Shape: {df.shape}")

Features added. Shape: (590540, 227)


In [13]:
#Split X and Y
X = df.drop(['TransactionID', 'isFraud'], axis=1)
y = df['isFraud']

print(f"X: {X.shape}")
print(f"y: {y.shape}  |  Fraud: {y.sum():,}")

X: (590540, 225)
y: (590540,)  |  Fraud: 20,663


In [14]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")

Train: 472,432  |  Test: 118,108


In [15]:
# Smote
print("Applying SMOTE... (3-5 mins)")

smote = SMOTE(sampling_strategy=0.1, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"After SMOTE — Fraud: {y_train_sm.sum():,}  Legit: {(y_train_sm==0).sum():,}")

Applying SMOTE... (3-5 mins)
After SMOTE — Fraud: 45,590  Legit: 455,902


In [16]:
# Save
with open(DRIVE_PATH + 'processed_data.pkl', 'wb') as f:
    pickle.dump({
        'X_train_sm': X_train_sm,
        'y_train_sm': y_train_sm,
        'X_test':     X_test,
        'y_test':     y_test,
        'X_train':    X_train,
        'y_train':    y_train
    }, f)

print("Saved! Move to Notebook 3.")

Saved! Move to Notebook 3.
